# 🚢 Imports 

In [12]:
import pandas as pd
import plotly.express as px
from types import SimpleNamespace

# 🕸️ Configs

In [ ]:
TABULAR_COLS = ["open", "high", "low", "close", "volume", "return_pct", "hl_spread_pct"]
SUFFIXES = {'ctgan': '20260305_133502', 'gmm': '20260305_140627', 'timegan': '20260305_151852'}

# 👣 Generate paths for the files

In [ ]:
REAL_PATHS = {f'{model}_real_data_path': f"outputs/real_data_{model}_{SUFFIXES[model]}.csv" for model in SUFFIXES}
SYNTHETIC_PATHS = {f'{model}_synthetic_data_path': f"outputs/synthetic_data_{model}_{SUFFIXES[model]}.csv" for model in SUFFIXES}
OTHER_PATHS = {
    'gmm_component_means_path': "outputs/gmm_component_means.csv",
    'gmm_component_correlations_path': "outputs/gmm_component_0_correlation_matrix.csv",
    'ctgan_training_log_path': "outputs/ctgan_training_log.csv"
}

PATHS = {**REAL_PATHS, **SYNTHETIC_PATHS, **OTHER_PATHS}  
PATHS 


p = SimpleNamespace(**PATHS)

print("PATHS and p NameSpace generated successfully")

# ⬇️ Read all CSVs into DataFrames

In [ ]:
gmm_real_data_df = pd.read_csv(p.gmm_real_data_path)
gmm_synthetic_data_df = pd.read_csv(p.gmm_synthetic_data_path)
ctgan_real_data_df = pd.read_csv(p.ctgan_real_data_path)
ctgan_synthetic_data_df = pd.read_csv(p.ctgan_synthetic_data_path)
timegan_real_data_df = pd.read_csv(p.timegan_real_data_path)
timegan_synthetic_data_df = pd.read_csv(p.timegan_synthetic_data_path)
gmm_component_means_df = pd.read_csv(p.gmm_component_means_path, index_col=0)
gmm_component_correlations_df = pd.read_csv(p.gmm_component_correlations_path)
ctgan_log_df = pd.read_csv(p.ctgan_training_log_path)

print(f"GMM Real Data || Len: {len(gmm_real_data_df)}")
print(f"GMM Synthetic Data || Len: {len(gmm_synthetic_data_df)}")
print(f"CTGAN Real Data || Len: {len(ctgan_real_data_df)}")
print(f"CTGAN Synthetic Data || Len: {len(ctgan_synthetic_data_df)}")

GMM Real Data || Len: 2514
GMM Synthetic Data || Len: 500
CTGAN Real Data || Len: 2514
CTGAN Synthetic Data || Len: 500


# 1️⃣ Compare return distributions [GMM]

In [ ]:
true_labels = ['True'] * len(gmm_real_data_df)
synthetic_labels = ['Synthetic'] * len(gmm_synthetic_data_df)
true_returns = pd.DataFrame({
    'return_pct': gmm_real_data_df['return_pct'],
    'label': true_labels
})
synthetic_returns = pd.DataFrame({
    'return_pct': gmm_synthetic_data_df['return_pct'],
    'label': synthetic_labels
})
returns_df = pd.concat([true_returns, synthetic_returns], ignore_index=True)

px.histogram(returns_df, x="return_pct", color="label", title="[GMM] Real vs Synthetic Data: Close Prices")

# 2️⃣ Compare distributions [CTGAN]

In [17]:
true_labels = ['True'] * len(ctgan_real_data_df)
synthetic_labels = ['Synthetic'] * len(ctgan_synthetic_data_df)
true_returns = pd.DataFrame({
    'return_pct': ctgan_real_data_df['return_pct'],
    'label': true_labels
})
synthetic_returns = pd.DataFrame({
    'return_pct': ctgan_synthetic_data_df['return_pct'],
    'label': synthetic_labels
})
returns_df = pd.concat([true_returns, synthetic_returns], ignore_index=True)

px.histogram(returns_df, x="return_pct", color="label", title="[CTGAN] Real vs Synthetic Data: Close Prices")

In [ ]:
# 3️⃣ Compare distributions [TIMEGAN]

In [47]:
gmm_component_means_df = gmm_component_means_df.reset_index()
gmm_component_means_df.columns = ['component'] + TABULAR_COLS
gmm_component_means_df.head()
gmm_component_means_df =gmm_component_means_df.melt(id_vars='component', value_vars = TABULAR_COLS)
px.bar(gmm_component_means_df, x='variable', y='value', color='component', barmode='group', title="GMM Component Means")


In [79]:
gmm_component_correlations_df.columns = ['column'] + TABULAR_COLS
fig=px.imshow(
    gmm_component_correlations_df[TABULAR_COLS].to_numpy(),
    x=TABULAR_COLS,
    y=TABULAR_COLS,
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="GMM Component 0 Correlation Matrix"
)

fig.update_xaxes(showgrid=True, gridcolor="red", gridwidth=10)
fig.update_yaxes(showgrid=True, gridcolor="red", gridwidth=10)

fig.show()

In [80]:
ctgan_log_df = ctgan_log_df.melt(id_vars='epoch', value_vars=['critic_loss', 'generator_loss'], var_name='loss_type', value_name='loss')
px.line(ctgan_log_df, x='epoch', y='loss', color='loss_type', markers=True, title="CTGAN Training: Critic and Generator Loss")

# Compare distributions [TIMEGAN]



In [5]:

true_labels = ['True'] * len(timegan_real_data_df)
synthetic_labels = ['Synthetic'] * len(timegan_synthetic_data_df)
true_returns = pd.DataFrame({
    'return_pct': timegan_real_data_df['return_pct'],
    'label': true_labels
})
synthetic_returns = pd.DataFrame({
    'return_pct': timegan_synthetic_data_df['return_pct'],
    'label': synthetic_labels
})
returns_df = pd.concat([true_returns, synthetic_returns], ignore_index=True)

px.histogram(returns_df, x="return_pct", color="label", title="[TimeGAN] Real vs Synthetic Data: Close Prices")